In [1]:
from collections import Counter

# --------------------------------------------------
# 1. Initial Vocabulary
# --------------------------------------------------

vocab = [
    "h",
    "p",
    "##u",
    "##g",
    "##s"
]

print("Initial Vocabulary:")
print(vocab)


# --------------------------------------------------
# 2. Training Words and Frequencies
# --------------------------------------------------

word_freq = {
    "hug": 2,
    "hugs": 1,
    "pug": 1
}

print("\nTraining Words:")
for word, freq in word_freq.items():
    print(word, "->", freq)


# --------------------------------------------------
# 3. Split Words into Tokens
# --------------------------------------------------

splits = {
    "hug": ["h", "##u", "##g"],
    "hugs": ["h", "##u", "##g", "##s"],
    "pug": ["p", "##u", "##g"]
}

print("\nInitial Splits:")
for word, tokens in splits.items():
    print(word, "->", tokens)


# --------------------------------------------------
# 4. Calculate Token Frequencies and Pair Frequencies
# --------------------------------------------------

token_freq = Counter()
pair_freq = Counter()

for word, freq in word_freq.items():

    tokens = splits[word]

    # Token frequency
    for token in tokens:
        token_freq[token] += freq

    # Pair frequency
    for i in range(len(tokens) - 1):
        pair = (tokens[i], tokens[i + 1])
        pair_freq[pair] += freq


print("\nToken Frequencies:")
for token, freq in token_freq.items():
    print(token, ":", freq)


print("\nPair Frequencies:")
for pair, freq in pair_freq.items():
    print(pair, ":", freq)


# --------------------------------------------------
# 5. Calculate WordPiece Scores
# --------------------------------------------------

scores = {}

for pair, freq in pair_freq.items():

    first = pair[0]
    second = pair[1]

    score = freq / (
        token_freq[first] * token_freq[second]
    )

    scores[pair] = score


print("\nWordPiece Scores:")

for pair, score in scores.items():
    print(pair, "=", round(score, 4))


# --------------------------------------------------
# 6. Find the Highest-Scoring Pair
# --------------------------------------------------

best_pair = max(
    scores,
    key=scores.get
)

print("\nBest Pair:")
print(best_pair)

print("Highest Score:", scores[best_pair])


# --------------------------------------------------
# 7. Merge the Best Pair
# --------------------------------------------------

def merge_pair(splits, pair):

    # Create new token
    new_token = pair[0] + pair[1].replace("##", "")

    for word in splits:

        tokens = splits[word]
        new_tokens = []

        i = 0

        while i < len(tokens):

            # Check adjacent pair
            if i < len(tokens) - 1:

                current_pair = (
                    tokens[i],
                    tokens[i + 1]
                )

                if current_pair == pair:

                    new_tokens.append(new_token)

                    i += 2
                    continue

            new_tokens.append(tokens[i])

            i += 1

        splits[word] = new_tokens

    return new_token


new_token = merge_pair(
    splits,
    best_pair
)

print("\nNew Token Created:")
print(new_token)

print("\nUpdated Splits:")

for word, tokens in splits.items():
    print(word, "->", tokens)


# --------------------------------------------------
# 8. Create Learned Vocabulary
# --------------------------------------------------

learned_vocab = {
    "h",
    "p",
    "##u",
    "##g",
    "##s",
    "hu",
    "hug",
    "##gs"
}

print("\nLearned Vocabulary:")
print(learned_vocab)


# --------------------------------------------------
# 9. WordPiece Tokenization
# --------------------------------------------------

def tokenize_word(word, vocab):

    tokens = []

    while len(word) > 0:

        found = False

        # Longest matching substring
        for i in range(len(word), 0, -1):

            part = word[:i]

            # Add ## for tokens inside the word
            if len(tokens) > 0:
                part = "##" + part

            if part in vocab:

                tokens.append(part)

                word = word[i:]

                found = True

                break

        # If no valid token is found
        if not found:
            return ["[UNK]"]

    return tokens


# --------------------------------------------------
# 10. Test Tokenization
# --------------------------------------------------

word = "hugs"

tokens = tokenize_word(
    word,
    learned_vocab
)

print("\nWord:")
print(word)

print("\nWordPiece Tokens:")
print(tokens)


# --------------------------------------------------
# 11. Vocabulary with Token IDs
# --------------------------------------------------

vocab_with_ids = {
    "[UNK]": 0,
    "h": 1,
    "p": 2,
    "##u": 3,
    "##g": 4,
    "##s": 5,
    "hu": 6,
    "hug": 7,
    "##gs": 8
}


# --------------------------------------------------
# 12. Convert Tokens to IDs
# --------------------------------------------------

ids = [
    vocab_with_ids[token]
    for token in tokens
]

print("\nTokens:")
print(tokens)

print("\nToken IDs:")
print(ids)


# --------------------------------------------------
# 13. Test Unknown Word
# --------------------------------------------------

unknown_word = "bum"

unknown_tokens = tokenize_word(
    unknown_word,
    learned_vocab
)

print("\nUnknown Word:")
print(unknown_word)

print("\nResult:")
print(unknown_tokens)

Initial Vocabulary:
['h', 'p', '##u', '##g', '##s']

Training Words:
hug -> 2
hugs -> 1
pug -> 1

Initial Splits:
hug -> ['h', '##u', '##g']
hugs -> ['h', '##u', '##g', '##s']
pug -> ['p', '##u', '##g']

Token Frequencies:
h : 3
##u : 4
##g : 4
##s : 1
p : 1

Pair Frequencies:
('h', '##u') : 3
('##u', '##g') : 4
('##g', '##s') : 1
('p', '##u') : 1

WordPiece Scores:
('h', '##u') = 0.25
('##u', '##g') = 0.25
('##g', '##s') = 0.25
('p', '##u') = 0.25

Best Pair:
('h', '##u')
Highest Score: 0.25

New Token Created:
hu

Updated Splits:
hug -> ['hu', '##g']
hugs -> ['hu', '##g', '##s']
pug -> ['p', '##u', '##g']

Learned Vocabulary:
{'##g', 'h', 'hu', '##s', '##gs', '##u', 'hug', 'p'}

Word:
hugs

WordPiece Tokens:
['hug', '##s']

Tokens:
['hug', '##s']

Token IDs:
[7, 5]

Unknown Word:
bum

Result:
['[UNK]']
